# 🌫️ Análisis Exploratorio de Datos (EDA) - Calidad del Aire en India
### Arquitectura Medallón: Excel (Red de Sensores) + API REST (Valores de Calidad del Aire)
---
Este notebook realiza un análisis exploratorio integral pero conciso combinando las capas procesadas en el pipeline ETL:
1. **Silver Sensores (Excel)**: Catálogo de estaciones y sensores activos en India.
2. **Silver Horario (API)**: Serie temporal horaria de contaminantes (PM2.5, PM10, NO2, SO2, CO, O3, AQI).
3. **Gold Consolidado**: Indicadores integrados por estación (cobertura + calidad de aire diaria + lecturas actuales).


## 1. Configuración y Carga de Datasets
Cargamos las librerías necesarias y los datasets limpios generados por el pipeline.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo visual moderno y limpio
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.autolayout"] = True

# Resolucion de rutas dinamica (funciona desde /notebooks o desde la raiz del proyecto)
base_dir = ".." if os.path.exists("../data") else "."
ruta_silver_sensores = f"{base_dir}/data/silver/silver_aire_india_sensores.csv"
ruta_silver_horario = f"{base_dir}/data/silver/silver_aire_india_horario.csv"
ruta_gold = f"{base_dir}/data/gold/gold_aire_india.csv"

df_sensores = pd.read_csv(ruta_silver_sensores)
df_horario = pd.read_csv(ruta_silver_horario, parse_dates=["date"])
df_gold = pd.read_csv(ruta_gold)

print(f"Silver Sensores (Excel): {len(df_sensores):,} filas x {df_sensores.shape[1]} columnas")
print(f"Silver Horario (API):    {len(df_horario):,} filas x {df_horario.shape[1]} columnas")
print(f"Gold Consolidado:        {len(df_gold):,} filas x {df_gold.shape[1]} columnas")

## 2. Inspección General del Dataset Gold
Visualizamos una muestra de las principales variables de cobertura y calidad del aire por estación.

In [ ]:
cols_muestra = [
    "location_name", "nivel_cobertura", "total_contaminantes",
    "aqi_promedio_dia", "pm2_5_promedio_dia", "pm10_promedio_dia",
    "categoria_aqi_predominante", "ranking_peor_aire"
]
df_gold[cols_muestra].head(8)

## 3. Diagnóstico de la Red de Sensores (Capa Silver - Excel)
Evaluamos el nivel de cobertura por estación y la disponibilidad de sensores por cada contaminante.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Nivel de cobertura de las estaciones
cobertura_counts = df_gold["nivel_cobertura"].value_counts()
sns.barplot(x=cobertura_counts.values, y=cobertura_counts.index, hue=cobertura_counts.index, palette="crest", legend=False, ax=ax1)
ax1.set_title("Nivel de Cobertura por Estación", fontsize=13, weight="bold")
ax1.set_xlabel("Número de Estaciones")
for i, v in enumerate(cobertura_counts.values):
    ax1.text(v + 5, i, f"{v} ({v/len(df_gold)*100:.1f}%)", va="center", fontsize=10)

# 2. Contaminantes mas monitoreados en la red
contaminantes_counts = df_sensores["parameter_std"].value_counts()
sns.barplot(x=contaminantes_counts.values, y=contaminantes_counts.index, hue=contaminantes_counts.index, palette="viridis", legend=False, ax=ax2)
ax2.set_title("Frecuencia de Contaminantes Monitoreados", fontsize=13, weight="bold")
ax2.set_xlabel("Número de Sensores Instalados")
for i, v in enumerate(contaminantes_counts.values):
    ax2.text(v + 10, i, f"{v:,}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

## 4. Estadísticas Descriptivas y Estado de la Calidad del Aire
Analizamos las métricas de los contaminantes y la distribución de categorías según el índice europeo (EAQI).

In [ ]:
# Tabla descriptiva de contaminantes principales
cols_metricas = [
    "pm2_5_promedio_dia", "pm10_promedio_dia", "no2_promedio_dia",
    "so2_promedio_dia", "co_promedio_dia", "o3_promedio_dia", "aqi_promedio_dia"
]
resumen = df_gold[cols_metricas].describe().T[["mean", "std", "min", "50%", "max"]]
resumen.columns = ["Media", "Desv. Est.", "Mínimo", "Mediana (P50)", "Máximo"]
display(resumen.round(2))

# Distribucion de categorias de calidad del aire
plt.figure(figsize=(9, 4.5))
orden_cat = ["Muy bueno", "Bueno", "Moderado", "Pobre", "Muy pobre", "Extremadamente pobre"]
paleta_semaforo = {
    "Muy bueno": "#2ecc71", "Bueno": "#27ae60", "Moderado": "#f1c40f",
    "Pobre": "#e67e22", "Muy pobre": "#e74c3c", "Extremadamente pobre": "#8e44ad"
}
cat_counts = df_gold["categoria_aqi_predominante"].value_counts().reindex(orden_cat).dropna()
colores = [paleta_semaforo.get(c, "#95a5a6") for c in cat_counts.index]

barras = plt.bar(cat_counts.index, cat_counts.values, color=colores, edgecolor="black", alpha=0.85)
plt.title("Distribución de la Calidad del Aire Predominante en India", fontsize=13, weight="bold")
plt.ylabel("Cantidad de Estaciones")
plt.xticks(rotation=20)
for bar in barras:
    altura = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, altura + 5, f"{int(altura)}", ha="center", weight="bold")
plt.show()

## 5. Ranking: Top 10 Estaciones con Peor Calidad del Aire
Identificamos las ubicaciones más críticas del país clasificadas por el índice AQI promedio.

In [ ]:
top10 = df_gold.sort_values("ranking_peor_aire").head(10)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=top10, x="aqi_promedio_dia", y="location_name", hue="location_name", palette="Reds_r", legend=False, ax=ax)
ax.set_title("Top 10 Estaciones con Peor Calidad del Aire (Mayor AQI)", fontsize=13, weight="bold")
ax.set_xlabel("Índice de Calidad del Aire (AQI Promedio Diario)")
ax.set_ylabel("Estación / Ubicación")
for i, (_, row) in enumerate(top10.iterrows()):
    ax.text(row["aqi_promedio_dia"] + 2, i, f"AQI {row['aqi_promedio_dia']:.1f} | PM2.5: {row['pm2_5_promedio_dia']:.1f}",
            va="center", fontsize=9, weight="bold")
plt.show()

## 6. Matriz de Correlación entre Contaminantes
Exploramos las relaciones estadísticas entre partículas finas ($PM_{2.5}, PM_{10}$) y gases contaminantes ($NO_2, SO_2, CO, O_3$).

In [ ]:
contaminantes = ["pm2_5", "pm10", "carbon_monoxide", "nitrogen_dioxide", "sulfur_dioxide", "ozone", "european_aqi"]
corr = df_horario[contaminantes].corr()

plt.figure(figsize=(8.5, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, linewidths=0.6)
plt.title("Matriz de Correlación de Contaminantes (Lecturas Horarias)", fontsize=13, weight="bold")
plt.show()

## 7. Dinámica Temporal: Ciclo Diario Promedio (24 Horas)
Analizamos cómo varía la concentración de contaminantes a lo largo del día para detectar picos horarios.

In [ ]:
perfil_horario = df_horario.groupby("hora")[["pm2_5", "pm10", "european_aqi", "ozone"]].mean()

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax2 = ax1.twinx()

line1 = ax1.plot(perfil_horario.index, perfil_horario["pm2_5"], color="#e74c3c", marker="o", label="PM2.5 (µg/m³)", linewidth=2)
line2 = ax1.plot(perfil_horario.index, perfil_horario["pm10"], color="#e67e22", marker="s", label="PM10 (µg/m³)", linewidth=2)
line3 = ax2.plot(perfil_horario.index, perfil_horario["european_aqi"], color="#2c3e50", linestyle="--", marker="^", label="AQI General", linewidth=2)

ax1.set_title("Evolución Promedio Horaria de Contaminantes (Ciclo Diario 24h)", fontsize=13, weight="bold")
ax1.set_xlabel("Hora del Día (Zona Horaria India IST)")
ax1.set_ylabel("Concentración Material Particulado (µg/m³)")
ax2.set_ylabel("Índice EAQI")
ax1.set_xticks(range(0, 24))

lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper left")
plt.show()

## 8. Distribución Geográfica de la Calidad del Aire
Mapeamos la ubicación espacial de las estaciones (Latitud y Longitud) junto con la severidad del AQI.

In [ ]:
plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    df_gold["longitude"], df_gold["latitude"],
    c=df_gold["aqi_promedio_dia"], cmap="YlOrRd",
    s=df_gold["pm2_5_promedio_dia"].clip(lower=20, upper=220),
    alpha=0.75, edgecolors="black", linewidth=0.5
)
cbar = plt.colorbar(scatter)
cbar.set_label("AQI Promedio Diario", rotation=270, labelpad=15)
plt.title("Dispersión Espacial de Estaciones en India\n(Color: AQI Promedio | Tamaño de Punto: PM2.5)", fontsize=13, weight="bold")
plt.xlabel("Longitud (°E)")
plt.ylabel("Latitud (°N)")
plt.show()

## 8.1 Comparación de Índices: Europeo (EAQI) vs. Nacional de India (NAQI - CPCB)
Comparamos el índice europeo devuelto por Open-Meteo frente al índice oficial de India (**NAQI - CPCB**, escala 0–500) y el contaminante dominante por estación.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
orden_naqi = ["Bueno", "Satisfactorio", "Moderado", "Pobre", "Muy pobre", "Severo"]
naqi_counts = df_gold["categoria_naqi_india"].value_counts().reindex(orden_naqi).dropna()
sns.barplot(x=naqi_counts.index, y=naqi_counts.values, hue=naqi_counts.index, palette="YlOrRd", legend=False, ax=ax1)
ax1.set_title("Distribución según Índice Oficial de India (NAQI - CPCB)", fontsize=12, weight="bold")
ax1.set_ylabel("Estaciones-Fecha")
dom_counts = df_gold["contaminante_dominante_naqi"].value_counts()
sns.barplot(x=dom_counts.values, y=dom_counts.index, hue=dom_counts.index, palette="magma", legend=False, ax=ax2)
ax2.set_title("Contaminante Dominante del NAQI (India CPCB)", fontsize=12, weight="bold")
plt.tight_layout()
plt.show()

## 8.2 Auditoría de Integración: Coordenadas Compartidas y Modelo CAMS vs. Estación Física
Dado que el modelo CAMS de Open-Meteo opera en una rejilla de ~0.1° (~11 km), las estaciones cercanas en una misma ciudad comparten punto de consulta (`id_punto_consulta`). Aquí inspeccionamos las celdas compartidas y la distancia (`distancia_modelo_km`) entre cada estación y el nodo del modelo.

In [ ]:
ruta_reporte = f"{base_dir}/data/gold/reporte_coordenadas_compartidas.csv"
df_rep = pd.read_csv(ruta_reporte)
print(f"Puntos de consulta únicos: {len(df_rep)} | Celdas compartidas (>1 estación): {(df_rep['valor_compartido']).sum()}")
print(f"Distancia promedio estación <-> nodo CAMS: {df_gold['distancia_modelo_km'].mean():.2f} km (máx: {df_gold['distancia_modelo_km'].max():.2f} km)")
display(df_rep[["id_punto_consulta", "n_estaciones_mismo_punto", "distancia_promedio_km", "location_names"]].head(8))

## 9. Conclusiones Finales y Diagnóstico Integral del EDA

A través de la arquitectura Medallón (Bronze -> Silver -> Gold), se logró transformar datos heterogéneos de sensores físicos (catálogo estático de estaciones) y mediciones atmosféricas en tiempo real/modelo (API) en un repositorio analítico confiable y consistente. El análisis exploratorio revela conclusiones contundentes sobre el estado de la calidad del aire en India:

---

### 1. El Material Particulado como Emergencia Sanitaria Predominante
El análisis estadístico y la matriz de correlación demuestran de manera inequívoca que el **PM2.5 y el PM10** son los principales impulsores del deterioro de la calidad del aire. Con una concentración promedio nacional de **46.89 µg/m³** para PM2.5 (triplicando el umbral diario recomendado por la OMS de 15 µg/m³) y valores máximos que superan los **119.12 µg/m³**, la mayor parte del territorio analizado se encuentra en rangos considerados **Pobres a Extremadamente Pobres** (AQI promedio de 71.94). A diferencia de los contaminantes gaseosos que presentan dispersión más rápida, las partículas suspendidas permanecen en el ambiente y representan el mayor riesgo de morbilidad respiratoria y cardiovascular en la población.

### 2. Dinámica Temporal: El Fenómeno del Ciclo Diario Bimodal
La serie horaria procesada en la capa Silver descubrió un ciclo diario recurrente con dos picos críticos de contaminación:
- **Pico Matutino (06:00 - 09:00 IST)**: Coincide con la hora punta del tráfico y el enfriamiento nocturno previo, generando una **inversión térmica** que atrapa los contaminantes cerca de la superficie.
- **Valle Vespertino y Pico de Ozono (13:00 - 16:00 IST)**: Conforme aumenta la temperatura y la altura de la capa de mezcla planetaria, las partículas se dispersan temporalmente; no obstante, la radiación solar activa la reacción fotoquímica del **Ozono troposférico (O3)**, que alcanza en este horario sus máximos diarios (>100 µg/m³).
- **Pico Nocturno (20:00 - 23:00 IST)**: Se observa un rebote severo del AQI debido a la reducción de los vientos, actividades industriales continuas y el tráfico de carga pesada.

### 3. Disparidad Geográfica y Puntos Críticos
El mapa de dispersión espacial evidencia una marcada asimetría regional:
- **Cinturón Norte (Llanura Indogangética, Delhi, Haryana, Punjab)**: Concentra los peores índices del país (top del ranking con AQI >160 y PM2.5 >100 µg/m³), exacerbado por factores topográficos de cuenca cerrada, alta densidad poblacional y actividad agrícola/industrial.
- **Zonas Costeras y Peninsulares (Sur/Suroeste)**: Registran niveles moderados y menores concentraciones de material particulado gracias a la circulación de brisas marinas y mayor ventilación natural.

### 4. Estado de la Red de Monitoreo y Recomendaciones
El diagnóstico del catálogo de estaciones (Excel) indica una red con buena madurez operativa, donde la mayoría de estaciones cuenta con capacidad para medir simultáneamente PM2.5 y PM10. Sin embargo, se identifican brechas en el despliegue sistemático de sensores para gases reactivos como **SO2** y **O3**, fundamentales para prever fenómenos de smog fotoquímico.

**Conclusión Final**:
La implementación de este pipeline no solo automatiza la ingesta y estandarización de datos multi-fuente, sino que sienta las bases para modelos predictivos de alerta temprana y políticas públicas focalizadas en las franjas horarias y regiones geográficas de mayor vulnerabilidad.